# Send a poem through a video

Read `media/poem.txt`, encode its bytes as data and Reed–Solomon parity QR codes, and embed them in a source video. Then recover the poem from a target video.


In [2]:
from pathlib import Path

import cv2
import numpy as np
import zxingcpp

from qr_solomon import QRConfig, QRDecoder, encode_message
from video_processor import VideoProcessor


In [ ]:
SOURCE_VIDEO = Path("potato_chip.mp4")
POEM_PATH = Path("media/poem.txt")
OUTPUT_VIDEO = Path("output/poem_embedded.mp4")

TARGET_VIDEO = OUTPUT_VIDEO

# Outer correction: parity_shards missing QR codes can be recovered.
# Inner correction: qr_error controls damage correction within each QR image.
CONFIG = QRConfig(
    shard_bytes=64,
    parity_shards=4,
    qr_version=10,
    qr_error="H",
)
QR_CHANGE_DELAY = 0.5  # Seconds per QR code; the sequence repeats.
QR_MODULE_SIZE = 8    # Pixels per QR module.
STRENGTH = 0.04       # Brightness offset as a fraction of 255.


## Read the poem and create QR codes

The file is read as bytes to preserve its exact UTF-8 contents and line endings. Each QR code carries one data or parity shard. The selected QR version determines the rendered size; reduce `QR_MODULE_SIZE` if it does not fit the source video.


In [4]:
poem_bytes = POEM_PATH.read_bytes()
qr_codes = encode_message(poem_bytes, CONFIG)
data_shards = len(qr_codes) - CONFIG.parity_shards

print(f"Poem: {len(poem_bytes)} bytes")
print(f"QR codes: {data_shards} data + {CONFIG.parity_shards} parity = {len(qr_codes)} total")
print(f"One full sequence: {len(qr_codes) * QR_CHANGE_DELAY:.2f} seconds")


Poem: 625 bytes
QR codes: 10 data + 4 parity = 14 total
One full sequence: 7.00 seconds


In [5]:
if SOURCE_VIDEO.resolve() == OUTPUT_VIDEO.resolve():
    raise ValueError("Choose different source and output video paths")
if QR_CHANGE_DELAY <= 0 or not np.isfinite(QR_CHANGE_DELAY):
    raise ValueError("QR_CHANGE_DELAY must be finite and greater than zero")
if type(QR_MODULE_SIZE) is not int or QR_MODULE_SIZE < 1:
    raise ValueError("QR_MODULE_SIZE must be a positive integer")

source = cv2.VideoCapture(str(SOURCE_VIDEO))
try:
    if not source.isOpened():
        raise RuntimeError(f"Could not open source video: {SOURCE_VIDEO}")
    fps = source.get(cv2.CAP_PROP_FPS)
    frame_count = source.get(cv2.CAP_PROP_FRAME_COUNT)
    width = int(source.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(source.get(cv2.CAP_PROP_FRAME_HEIGHT))
finally:
    source.release()

if not np.isfinite(fps) or fps <= 0:
    raise ValueError("Source video has an invalid frame rate")
if QR_CHANGE_DELAY < 1 / fps:
    raise ValueError("QR_CHANGE_DELAY must allow at least one source frame per QR code")
qr_width, qr_height = qr_codes[0].symbol_size(scale=QR_MODULE_SIZE, border=4)
if qr_width > width or qr_height > height:
    raise ValueError(f"QR size {qr_width}x{qr_height} exceeds video size {width}x{height}")
if frame_count <= 0 or (frame_count - 1) / fps < (len(qr_codes) - 1) * QR_CHANGE_DELAY:
    raise ValueError("Source video is too short for a full QR sequence; use a longer video or shorter delay")

processor = VideoProcessor(str(SOURCE_VIDEO), qr_codes)
processor.QR_CHANGE_DELAY = QR_CHANGE_DELAY
processor.QR_MODULE_SIZE = QR_MODULE_SIZE
processor.STRENGTH = STRENGTH
processor.output(str(OUTPUT_VIDEO))
print(f"Created: {OUTPUT_VIDEO}")


Opened: True
FPS: 30.0
Frames: 2707.0


ffmpeg version n9.0.2 Copyright (c) 2000-2026 the FFmpeg developers
  built with gcc 16 (GCC)
  configuration: --prefix=/usr --disable-debug --disable-static --disable-stripping --enable-amf --enable-avisynth --enable-cuda-llvm --enable-lto --enable-fontconfig --enable-frei0r --enable-gmp --enable-gnutls --enable-gpl --enable-ladspa --enable-lcms2 --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libdav1d --enable-libdrm --enable-libdvdnav --enable-libdvdread --enable-libfreetype --enable-libfribidi --enable-libgsm --enable-libharfbuzz --enable-libiec61883 --enable-libjack --enable-libjxl --enable-libmodplug --enable-libmp3lame --enable-libopencore_amrnb --enable-libopencore_amrwb --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libplacebo --enable-libpulse --enable-librav1e --enable-librsvg --enable-librubberband --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtheora --ena

Saved 60.00 FPS video to output/poem_embedded.mp4
Created: output/poem_embedded.mp4


[out#0/mp4 @ 0x5569f5f04800] video:205729KiB audio:1409KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.050963%
frame= 5404 fps=0.0 q=-1.0 Lsize=  207244KiB time=00:01:30.21 bitrate=18817.8kbits/s speed= 364x elapsed=0:00:00.24    


## Recover the message from a target video

Each scanned QR packet is pushed into a fresh `QRDecoder`. Duplicate packets do not advance recovery. Scanning stops when the decoder returns a complete message.


In [10]:
cap = cv2.VideoCapture(str(TARGET_VIDEO))
decoder = QRDecoder()
previous = None
recovered_message = None
frames_read = 0
packets_read = 0

try:
    if not cap.isOpened():
        raise RuntimeError(f"Could not open target video: {TARGET_VIDEO}")
    print(f"Reading: {TARGET_VIDEO}")

    while recovered_message is None:
        ok, frame = cap.read()
        if not ok:
            break
        frames_read += 1
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if previous is not None:
            difference = previous.astype(np.int16) - gray.astype(np.int16)
            qr_image = np.where(difference > 0, 0, 255).astype(np.uint8)

            for candidate in (qr_image, cv2.bitwise_not(qr_image)):
                results = zxingcpp.read_barcodes(
                    candidate,
                    formats=zxingcpp.BarcodeFormat.QRCode,
                )
                for result in results:
                    packets_read += 1
                    recovered_message = decoder.push(result.bytes)
                    if recovered_message is not None:
                        break
                if recovered_message is not None:
                    print(recovered_message.decode("utf-8"))
                    decoder.reset()
                    break
        previous = gray
finally:
    cap.release()

if recovered_message is not None:
    print(f"Recovered {len(recovered_message)} bytes after {frames_read} frames.")
    print()
    print(recovered_message.decode("utf-8"))


Reading: output/poem_embedded.mp4
by Manuel Gutiérrez Nájera

I want to die as the day declines, 
at high sea and facing the sky, 
while agony seems like a dream 
and my soul like a bird that can fly. 

To hear not, at this last moment, 
once alone with sky and sea, 
any more voices nor weeping prayers 
than the majestic beating of the waves. 

To die when the sad light retires 
its golden network from the green waves 
to be like the sun that slowly expires; 
something very luminous that fades. 

To die, and die young, before 
fleeting time removes the gentle crown, 
while life still says: "I'm yours" 
though we know with our hearts that she lies. 

Recovered 625 bytes after 272 frames.

by Manuel Gutiérrez Nájera

I want to die as the day declines, 
at high sea and facing the sky, 
while agony seems like a dream 
and my soul like a bird that can fly. 

To hear not, at this last moment, 
once alone with sky and sea, 
any more voices nor weeping prayers 
than the majestic beating of the